In [1]:
import cutlass 
import torch 
import cutlass.cute as cute 
from cutlass.cute.runtime import from_dlpack

In [2]:
@cute.kernel 
def tiled_mma_kern(gA:cute.Tensor, gB:cute.Tensor,gC:cute.Tensor, tiled_mma: cute.TiledMma):
  t,_,_ = cute.arch.thread_idx()
  thread_mma = tiled_mma.get_slice(t)
  thread_slice_A = thread_mma.partition_A(gA)
  thread_slice_B = thread_mma.partition_B(gB)
  thread_slice_C = thread_mma.partition_C(gC)
  tCrA = tiled_mma.make_fragment_A(thread_slice_A)
  tCrB = tiled_mma.make_fragment_B(thread_slice_B)
  tCrC = tiled_mma.make_fragment_C(thread_slice_C)
  tCrC.fill(0.0)
  cute.autovec_copy(thread_slice_A, tCrA)
  cute.autovec_copy(thread_slice_B, tCrB)
  cute.gemm(tiled_mma, tCrC, tCrA[None,None,0], tCrB[None,None,0], tCrC)
  cute.autovec_copy(tCrC, thread_slice_C)


In [3]:
@cute.jit 
def tiled_mma_launcher(gA:cute.Tensor, gB:cute.Tensor, gC:cute.Tensor): 
  op = cute.nvgpu.warp.MmaF16BF16Op(
    cutlass.BFloat16,          # fp16
    cutlass.Float32,        # fp32
    (16, 8, 16)      # mma_inst_mnk
  ) 
  atom_layout = (1,1,1)
  tC = cute.make_layout(atom_layout)
  tiled_mma = cute.make_tiled_mma(op, tC)
  tiled_mma_kern(gA,gB,gC,tiled_mma).launch(grid = [1,1,1], block =[32,1,1])
    

In [ ]:
M = 16 
N = 8 
K = 16

a = torch.randn(M, K, device="cuda", dtype=torch.bfloat16)
b = torch.randn(N, K, device="cuda", dtype=torch.bfloat16)
c = torch.zeros(M, N, device="cuda", dtype=torch.float32)
print(c)
a_ = from_dlpack(a, assumed_align=16)
b_ = from_dlpack(b, assumed_align=16)
c_ = from_dlpack(c, assumed_align=16)

tiled_mma_kern_ = cute.compile(tiled_mma_launcher, a_, b_, c_)
tiled_mma_kern_(a_, b_, c_)
print(c)
print(torch.testing.assert_close(c, torch.matmul(a.float(),b.transpose(0,1).float())))

print(torch.matmul(a.float(),b.transpose(0,1).float()))

tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]], device='cuda:0')
tensor([[-1.1090e+01,  1.9592e+00, -4.0001e+00, -1.3567e+00, -1.2057e+01,
          3.1086e+00, -6.4366e+00, -1.1138e+00],
        [-3.1464e-01,  3.4636e+00,  2.6155e-01, -2.8132e+00, -2.9871e+00,
         -1.3826e+00,  1.2869e+00, -3.6952e+00],
        [-8.7751e+00,  6.7177e-01, -3.1971e+00,  1.3896e+00, -5.